In [6]:
import numpy as np
import torch
from torch import torch_version

%pprint # to print arrays on one line

Pretty printing has been turned ON


In [7]:
from tokenizers import Tokenizer
from tokenizers.implementations import ByteLevelBPETokenizer

In [28]:
tokenizer = ByteLevelBPETokenizer()
tokenizer.train(["C:\\dev\\tutorials\\LLM2\\gpt\\makemore\\data\\TinyStoriesV2-GPT4-train.txt"], vocab_size=10_000, special_tokens=["<|endoftext|>"])
tokenizer.save("data/tokenizer.json")

In [8]:
tokenizer = Tokenizer.from_file("data/tokenizer.json")

In [29]:
#testing the tokenizer
text = "Hello, y'all! How are you 😁 ?<|endoftext|>"
output = tokenizer.encode(text)
print(output.tokens)
print(tokenizer.decode(output.ids, skip_special_tokens=False))
print(f"Compression ratio: {len(text) / len(output.ids)}")

['Hello', ',', 'Ġy', "'", 'all', '!', 'ĠHow', 'Ġare', 'Ġyou', 'Ġ', 'ð', 'Ł', 'ĺ', 'ģ', 'Ġ', '?', '<|endoftext|>']
Hello, y'all! How are you 😁 ?<|endoftext|>
Compression ratio: 2.4705882352941178


In [8]:
# Save tokenized data into .bin file
from pathlib import Path

def save(bin_file: str, batch: list[str]):
  with open(bin_file, mode="ab") as f:
    encoded = tokenizer.encode_batch(batch)
    np.array([i for enc in encoded for i in enc.ids], dtype=np.uint16).tofile(f)

def tokenize_file(filename: str) :
  bin_file = str(Path(filename).with_suffix(".bin"))
  Path(bin_file).unlink(missing_ok=True)
  with open(filename, encoding="utf-8") as file:
    batch = []
    cur_story = []
    for line in file:
      line = line.rstrip('\n')
      cur_story.append(line)
      if line.endswith("<|endoftext|>"):
        batch.append("\n".join(cur_story))
        cur_story = []
        if len(batch) > 10_000:
          save(bin_file, batch)
          batch = []
    if cur_story:
      batch.append("\n".join(cur_story))
    if batch:
      save(bin_file, batch)

In [9]:
tokenize_file("C:\\dev\\tutorials\\LLM2\\gpt\\makemore\\data\\TinyStoriesV2-GPT4-valid.txt")

In [10]:
#validate the saved tokenized data
data = np.fromfile("C:\\dev\\tutorials\\LLM2\\gpt\\makemore\\data\\TinyStoriesV2-GPT4-valid.bin", dtype=np.uint16)
text = tokenizer.decode(data[:1000], skip_special_tokens=False)
text

'u don\'t have to be scared of the loud dog, I\'ll protect you". The mole felt so safe with the little girl. She was very kind and the mole soon came to trust her. He leaned against her and she kept him safe. The mole had found his best friend.\n<|endoftext|>Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad.\nTom asked his friend, Sam, to help him search for the ball. They looked high and low, but they could not find the ball. Tom said, "I think my ball fell into the pit."\nSam and Tom went close to the pit. They were scared, but they wanted to find the red ball. They looked into the pit, but it was too dark to see. Tom said, "We must go in and search for my ball."\nThey went into the pit to search. It was dark and scary. They could not find the ball. They tried to get out, but the pit was too deep. Tom and Sam were stuck in the pit. They called for help, but no one could 

In [20]:
b = tokenizer.encode("""
It was a sunny day and two friends decided to travel. They were so happy to be able to take a journey together. One had a big tire and the other had a little tire. They rolled the tires down the hill and across the meadow, giggling and laughing.
Their journey took them past a number of houses and animals, they even stopped to feed a cute little squirrel. After a while the sun began to disappear and soon it was time to head home.
The two friends rolled the tires all the way back to the beginning of their journey. By the time they arrived home, their faces were bright and their hearts were full of joy. They said goodbye to one another feeling very happy after their fun journey.
<|endoftext|>
""").ids

print(len(b))
# print(tokenizer.decode(tensor.tolist(), skip_special_tokens=False))

151
